## Install depedencies

In [ ]:
# %%capture --no-display
# !pip install newsapi-python
# !pip install -U langchain-community
# !pip install rouge_score
# !pip install tiktoken
# !pip install faiss-cpu
# !pip install sentence-transformers
# !pip install bertopic
# !pip install fuzzywuzzy
# !pip install pydub
# !pip install librosa
# !pip install rank_bm25 nltk

In [2]:
import re
from sentence_transformers import SentenceTransformer, util
import statistics
import os
import requests
import pandas as pd
from datetime import datetime, timedelta

# from langchain.embeddings import HuggingFaceEmbeddings
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
import openai
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize

In [ ]:
import os
import openai

# Set your API key (or use environment variable OPENAI_API_KEY)
openai.api_key = os.getenv("OPENAI_API_KEY", "your-api-key")

In [4]:
# Helper function to display markdown
from IPython.display import Markdown, display
def md(text):
    display(Markdown(text))

## Web scrape Apple

In [5]:
import requests
from bs4 import BeautifulSoup
import os

def fetch_and_save_transcript(url, filename='apple_q4_2024_transcript.txt'):
    # Send a GET request to the URL
    response = requests.get(url)
    response.raise_for_status()

    # Parse the HTML content
    soup = BeautifulSoup(response.text, 'html.parser')

    # Find the article-body div
    article_body = soup.find('div', class_='article-body')
    if not article_body:
        raise ValueError("Could not find the article body.")

    # Remove all h2 tags from the article body
    for h2 in article_body.find_all('h2'):
        h2.decompose()

    # Extract all paragraph text
    paragraphs = article_body.find_all('p')
    transcript = '\n'.join([para.get_text() for para in paragraphs])

    # Save to a .txt file
    with open(filename, 'w', encoding='utf-8') as file:
        file.write(transcript)

    print(f"Transcript saved to '{filename}'")

# URL of the earnings call transcript
url = 'https://www.fool.com/earnings/call-transcripts/2024/10/31/apple-aapl-q4-2024-earnings-call-transcript/'

# Run the function
fetch_and_save_transcript(url)

Transcript saved to 'apple_q4_2024_transcript.txt'


In [6]:
file_path = "apple_q4_2024_transcript.txt"

with open(file_path, 'r', encoding='utf-8') as file:
    transcript = file.read()

transcript

"Image source: The Motley Fool.\nApple (AAPL -0.19%)Q4 2024 Earnings CallOct 31, 2024, 5:00 p.m. ET\nSuhasini Chandramouli -- Director, Investor Relations\nGood afternoon, and welcome to the Apple Q4 fiscal year 2024 earnings conference call. My name is Suhasini Chandramouli, director of investor relations. Today's call is being recorded. Speaking first today are Apple's CEO, Tim Cook; and CFO, Luca Maestri; and they'll be joined by Kevan Parekh, vice president of financial planning and analysis.\nAfter that, we'll open the call to questions from analysts. Please note that some of the information you'll hear during our discussion today will consist of forward-looking statements, including, without limitation, those regarding revenue, gross margin, operating expenses, other income and expense, taxes, capital allocation, and future business outlook, including the potential impact of macroeconomic conditions on the company's business and results of operations. These statements involve ris

In [7]:
import re
import json

def chunk_transcript(text):
    # Pattern to detect speakers including Operator
    speaker_pattern = re.compile(r'^(?:[A-Z][a-z]+(?: [A-Z][a-z]+)* -- .+|Operator)$', re.MULTILINE)
    chunks = []

    matches = list(speaker_pattern.finditer(text))
    pending_question = None
    question_speaker = None
    chunk_id = 1

    for i, match in enumerate(matches):
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        speaker_line = match.group(0).strip()
        speaker_text = text[start:end].strip().replace('\n', ' ')

        # Skip empty text
        if not speaker_text:
            continue

        # Skip Operator chunks
        if speaker_line == "Operator":
            continue

        # Store analyst question temporarily
        if "Analyst" in speaker_line:
            pending_question = speaker_text
            question_speaker = speaker_line
            continue

        # Build final chunk with or without question
        chunk = {
            "chunk_id": f"chunk_{chunk_id:03}",
            "speaker": speaker_line,
            "text": speaker_text,
            "question": pending_question if pending_question else ""
        }

        chunks.append(chunk)
        chunk_id += 1
        pending_question = None  # Clear after attaching to next answer

    return chunks


# Example usage
chunks = chunk_transcript(transcript)

# Save to JSON
with open("apple_q4_2024_chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2)

print(chunks[0])

{'chunk_id': 'chunk_001', 'speaker': 'Suhasini Chandramouli -- Director, Investor Relations', 'text': "Good afternoon, and welcome to the Apple Q4 fiscal year 2024 earnings conference call. My name is Suhasini Chandramouli, director of investor relations. Today's call is being recorded. Speaking first today are Apple's CEO, Tim Cook; and CFO, Luca Maestri; and they'll be joined by Kevan Parekh, vice president of financial planning and analysis. After that, we'll open the call to questions from analysts. Please note that some of the information you'll hear during our discussion today will consist of forward-looking statements, including, without limitation, those regarding revenue, gross margin, operating expenses, other income and expense, taxes, capital allocation, and future business outlook, including the potential impact of macroeconomic conditions on the company's business and results of operations. These statements involve risks and uncertainties that may cause actual results or 

In [9]:
chunks

[{'chunk_id': 'chunk_001',
  'speaker': 'Suhasini Chandramouli -- Director, Investor Relations',
  'text': "Good afternoon, and welcome to the Apple Q4 fiscal year 2024 earnings conference call. My name is Suhasini Chandramouli, director of investor relations. Today's call is being recorded. Speaking first today are Apple's CEO, Tim Cook; and CFO, Luca Maestri; and they'll be joined by Kevan Parekh, vice president of financial planning and analysis. After that, we'll open the call to questions from analysts. Please note that some of the information you'll hear during our discussion today will consist of forward-looking statements, including, without limitation, those regarding revenue, gross margin, operating expenses, other income and expense, taxes, capital allocation, and future business outlook, including the potential impact of macroeconomic conditions on the company's business and results of operations. These statements involve risks and uncertainties that may cause actual result

## Audio

In [ ]:
import os
from pydub import AudioSegment
import nltk
from nltk.tokenize import word_tokenize
from pprint import pprint

nltk.download('punkt_tab')


# Full audio file path
# audio_file_path = "aapl_q_4_2024_10_31_earnings_summary.mp3"
audio_file_path = "/content/drive/MyDrive/DSA4265/Project/aapl_q_4_2024_10_31_earnings_summary.mp3"

# Load the full audio with pydub
full_audio = AudioSegment.from_file(audio_file_path, format="mp3")
total_audio_ms = len(full_audio)  # duration in milliseconds

# Function to count words in a text
def count_words(text):
    return len(word_tokenize(text))

# Total word count across all chunks
total_words = sum(count_words(chunk['text']) for chunk in chunks)

current_start_ms = 0
for chunk in chunks:
    wc = count_words(chunk['text'])
    # Calculate duration for this chunk (proportional)
    chunk_duration_ms = int((wc / total_words) * total_audio_ms)
    chunk['start_time'] = current_start_ms
    chunk['end_time'] = current_start_ms + chunk_duration_ms

    # Extract and save the audio segment for the chunk
    audio_segment = full_audio[chunk['start_time']:chunk['end_time']]
    out_dir = "audio_chunks"
    os.makedirs(out_dir, exist_ok=True)
    out_filename = f"{chunk['chunk_id']}.mp3"
    out_path = os.path.join(out_dir, out_filename)
    audio_segment.export(out_path, format="mp3")
    chunk['audio_file'] = out_path

    current_start_ms += chunk_duration_ms

pprint(chunks)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


[{'audio_file': 'audio_chunks/chunk_001.mp3',
  'chunk_id': 'chunk_001',
  'end_time': 144393,
  'question': '',
  'speaker': 'Suhasini Chandramouli -- Director, Investor Relations',
  'start_time': 0,
  'text': 'Good afternoon, and welcome to the Apple Q4 fiscal year 2024 '
          'earnings conference call. My name is Suhasini Chandramouli, '
          "director of investor relations. Today's call is being recorded. "
          "Speaking first today are Apple's CEO, Tim Cook; and CFO, Luca "
          "Maestri; and they'll be joined by Kevan Parekh, vice president of "
          "financial planning and analysis. After that, we'll open the call to "
          'questions from analysts. Please note that some of the information '
          "you'll hear during our discussion today will consist of "
          'forward-looking statements, including, without limitation, those '
          'regarding revenue, gross margin, operating expenses, other income '
          'and expense, taxes, cap

In [ ]:
import librosa
import numpy as np

def extract_audio_features(audio_path, sr=16000):
    """
    Extracts several audio features from an audio file.

    Parameters:
        audio_path (str): Path to the audio file.
        sr (int): Sampling rate to use for loading audio.

    Returns:
        dict: A dictionary containing average MFCCs, mean pitch, and mean RMS energy.
    """
    # Load the audio segment using librosa.
    y, sr = librosa.load(audio_path, sr=sr)

    # Compute 13 MFCCs and take the mean across time for each coefficient.
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfccs_avg = np.mean(mfccs, axis=1).tolist()

    # Use librosa.pyin to estimate pitch (f0). fmin and fmax defined for typical speech range.
    f0, voiced_flag, voiced_prob = librosa.pyin(
        y,
        fmin=librosa.note_to_hz('C2'),
        fmax=librosa.note_to_hz('C7')
    )
    # Compute mean pitch over voiced regions (ignore unvoiced).
    if f0 is not None and np.any(voiced_flag):
        pitch_mean = float(np.mean(f0[voiced_flag]))
    else:
        pitch_mean = None

    # Calculate Root Mean Square (RMS) energy and compute the mean value.
    rms = librosa.feature.rms(y=y)
    rms_mean = float(np.mean(rms))

    return {
        'mfccs_avg': mfccs_avg,
        'pitch_mean': pitch_mean,
        'rms_mean': rms_mean
    }

# Example: Update each chunk by adding audio features.
# 'chunks' is a list of dictionaries, each with an 'audio_file' key.
for chunk in chunks:
    # Extract features from the audio segment corresponding to this chunk.
    features = extract_audio_features(chunk['audio_file'])
    # Add the extracted features to the chunk dictionary.
    chunk['audio_features'] = features

# Verify the updated chunks by printing them.
from pprint import pprint
pprint(chunks)

[{'audio_features': {'mfccs_avg': [-313.2426452636719,
                                   100.96583557128906,
                                   1.7491894960403442,
                                   7.238378047943115,
                                   -9.24455738067627,
                                   -12.993865013122559,
                                   -22.510822296142578,
                                   -28.745073318481445,
                                   -13.357366561889648,
                                   -15.916699409484863,
                                   -15.462661743164062,
                                   -10.061859130859375,
                                   -9.204828262329102],
                     'pitch_mean': 246.02537482723963,
                     'rms_mean': 0.06433051824569702},
  'audio_file': 'audio_chunks/chunk_001.mp3',
  'chunk_id': 'chunk_001',
  'end_time': 144393,
  'question': '',
  'speaker': 'Suhasini Chandramouli -- Director, Investo

Just run this:

In [5]:
import json
json_path = "apple_q4_2024_AUDIO_chunks.json"

# Open and load the JSON file
with open(json_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)

In [6]:
chunks

[{'chunk_id': 'chunk_001',
  'speaker': 'Suhasini Chandramouli -- Director, Investor Relations',
  'text': "Good afternoon, and welcome to the Apple Q4 fiscal year 2024 earnings conference call. My name is Suhasini Chandramouli, director of investor relations. Today's call is being recorded. Speaking first today are Apple's CEO, Tim Cook; and CFO, Luca Maestri; and they'll be joined by Kevan Parekh, vice president of financial planning and analysis. After that, we'll open the call to questions from analysts. Please note that some of the information you'll hear during our discussion today will consist of forward-looking statements, including, without limitation, those regarding revenue, gross margin, operating expenses, other income and expense, taxes, capital allocation, and future business outlook, including the potential impact of macroeconomic conditions on the company's business and results of operations. These statements involve risks and uncertainties that may cause actual result

### Audio Feature Engineering

In [7]:
import numpy as np
from pprint import pprint

# --- Step 1: Collect pitch and RMS features from all chunks for normalization ---
pitch_list = [chunk['audio_features']['pitch_mean'] for chunk in chunks if 'audio_features' in chunk]
rms_list   = [chunk['audio_features']['rms_mean'] for chunk in chunks if 'audio_features' in chunk]

pitch_global_mean = np.mean(pitch_list)
pitch_global_std  = np.std(pitch_list)
rms_global_mean   = np.mean(rms_list)
rms_global_std    = np.std(rms_list)

# --- Step 2: Feature engineering for each chunk ---
for chunk in chunks:
    if 'audio_features' in chunk:
        features = chunk['audio_features']

        # Standardize the pitch and rms values (z-score normalization)
        pitch = features.get('pitch_mean', 0)
        rms   = features.get('rms_mean', 0)

        pitch_z = (pitch - pitch_global_mean) / (pitch_global_std + 1e-8)
        rms_z   = (rms - rms_global_mean) / (rms_global_std + 1e-8)

        features['pitch_z'] = pitch_z
        features['rms_z']   = rms_z

        # Compute a composite intensity score based on the normalized values.
        # For instance, a simple approach:
        composite_intensity = abs(pitch_z) + abs(rms_z)
        features['composite_intensity'] = composite_intensity

        # Process MFCCs: convert list to numpy array
        mfccs = np.array(features.get('mfccs_avg', []))
        if mfccs.size > 0:
            # Mean absolute value of MFCCs can indicate overall spectral magnitude.
            mfccs_mean_abs = float(np.mean(np.abs(mfccs)))
            # Variance can indicate spectral spread or diversity.
            mfccs_variance = float(np.var(mfccs))
        else:
            mfccs_mean_abs = None
            mfccs_variance = None

        features['mfccs_mean_abs'] = mfccs_mean_abs
        features['mfccs_variance'] = mfccs_variance

# Print the updated chunks with engineered audio features
pprint(chunks)

[{'audio_features': {'composite_intensity': 1.8424681459880947,
                     'mfccs_avg': [-313.2426452636719,
                                   100.96583557128906,
                                   1.7491894960403442,
                                   7.238378047943115,
                                   -9.24455738067627,
                                   -12.993865013122559,
                                   -22.510822296142578,
                                   -28.745073318481445,
                                   -13.357366561889648,
                                   -15.916699409484863,
                                   -15.462661743164062,
                                   -10.061859130859375,
                                   -9.204828262329102],
                     'mfccs_mean_abs': 43.13029088423802,
                     'mfccs_variance': 7837.010109531141,
                     'pitch_mean': 246.02537482723963,
                     'pitch_z': 1.8353417528

In [8]:
chunks[0]

{'chunk_id': 'chunk_001',
 'speaker': 'Suhasini Chandramouli -- Director, Investor Relations',
 'text': "Good afternoon, and welcome to the Apple Q4 fiscal year 2024 earnings conference call. My name is Suhasini Chandramouli, director of investor relations. Today's call is being recorded. Speaking first today are Apple's CEO, Tim Cook; and CFO, Luca Maestri; and they'll be joined by Kevan Parekh, vice president of financial planning and analysis. After that, we'll open the call to questions from analysts. Please note that some of the information you'll hear during our discussion today will consist of forward-looking statements, including, without limitation, those regarding revenue, gross margin, operating expenses, other income and expense, taxes, capital allocation, and future business outlook, including the potential impact of macroeconomic conditions on the company's business and results of operations. These statements involve risks and uncertainties that may cause actual results o

## RAG

### Convert into FAISS Database for semantic search

In [39]:
import json
from langchain.schema import Document
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
from langchain.embeddings import HuggingFaceEmbeddings


# Convert documents
documents = [
    Document(
        page_content=entry["text"],
        metadata={
            "chunk_id": entry["chunk_id"],
            "speaker": entry["speaker"],
            "question": entry["question"],
            "audio_features": entry["audio_features"]
        }
    )
    for entry in chunks if entry["text"].strip()
]

# Initialize embedding model
# embedding_model = OpenAIEmbeddings(model="text-embedding-ada-002")
hf_embeddings = HuggingFaceEmbeddings(
    model_name="best_fine_tuned_earnings_retriever",
    model_kwargs={"device": "cpu"},             # e.g. "cuda" or "mps"
    encode_kwargs={"normalize_embeddings": True} # whether to L2‑normalize outputs
)


# Build FAISS index
# faiss_index = FAISS.from_documents(documents, embedding_model)
faiss_index = FAISS.from_documents(documents, hf_embeddings)

## Using Cross Encoder

In [10]:
from sentence_transformers import CrossEncoder

# Load the cross-encoder model
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

## Hybrid Retrieval and Cross Encoder Reranking


# Hybrid Retrieval (BM25 + FAISS)
Keyword-based search like BM25 is precise but lacks semantic understanding, while dense vector search like FAISS captures meaning but often returns irrelevant results. Neither approach alone is enough for Retrieval-Augmented Generation (RAG). Hybrid retrieval—combining BM25 and FAISS—bridges this gap. BM25 filters results using exact term matching, while FAISS refines them by identifying deeper semantic connections. Together, they enhance precision and contextual relevance

#  Rerank documents
This function re-ranks retrieved documents using a cross-encoder, which evaluates each (query, document) pair for relevance.
It first prepares inputs as pairs: [query, doc]
Then uses the cross_encoder.predict() method to assign a relevance score to each pair
After scoring, it attaches those scores to the docs and sorts them by score (highest first)
Finally, it returns the reordered list of documents, now ranked by the cross-encoder’s more accurate assessment of relevance

# Combined pipleline

It combines speed (bi-encoder) and accuracy (cross-encoder) — a common best practice in modern RAG pipelines.

## Sequentially
1. BM25 phase: retrieve top‑k1 lexical matches

2. Embedding phase (FAISS): retrieve top‑k2 semantic matches

3. Merge & De‑dupe these two lists (preserving score order)

4. Cross‑encode on the merged list to get final ranking

In [40]:
def retrieve_relevant_docs(query, vector_store, k=5):
    # Retrieve top-k similar document chunks for the query
    # Semantic simiarity search using the vector store
    # This will return the top-k most relevant documents based on the query
    retrieved_docs = vector_store.similarity_search(query, k=k)
    return retrieved_docs


def rerank_documents(query, retrieved_docs, cross_encoder):
    # Prepare the inputs for the cross-encoder
    cross_encoder_inputs = [[query, doc.page_content] for doc in retrieved_docs]

    # Compute relevance scores
    relevance_scores = cross_encoder.predict(cross_encoder_inputs)

    # Attach scores to documents
    pairs_list = []
    for idx, doc in enumerate(retrieved_docs):
        pairs_list.append((doc, relevance_scores[idx]))

    # Sort documents by relevance score in descending order
    sorted_docs = sorted(pairs_list, key=lambda x: x[1], reverse=True)

    # Final output
    reranked_docs = [doc for doc, _ in sorted_docs]

    return reranked_docs

In [41]:
from rank_bm25 import BM25Okapi
import nltk
import numpy as np

# Download tokenizer data
nltk.download("punkt")

# Extract the raw text corpus
corpus = [doc.page_content for doc in documents]

# Tokenize each document (lowercased)
tokenized_corpus = [nltk.word_tokenize(text.lower()) for text in corpus]

# Instantiate BM25
bm25 = BM25Okapi(tokenized_corpus)

[nltk_data] Downloading package punkt to C:\Users\Goh Ming
[nltk_data]     Wee\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [42]:
def retrieve_bm25(query: str, k: int = 5):
    # Tokenize the query
    tokenized_query = nltk.word_tokenize(query.lower())
    # Get BM25 scores for every document
    scores = bm25.get_scores(tokenized_query)
    # Pick top‑k indices
    top_indices = np.argsort(scores)[::-1][:k]
    # Return the corresponding Document objects
    return [documents[i] for i in top_indices]

In [43]:
def hybrid_retrieve(query, k1=10, k2=10, final_k=5, vector_store=None, cross_encoder=None):      
    # BM25 retrieval
    bm25_docs = retrieve_bm25(query, k=k1)

    # Embedding retrieval
    emb_docs = retrieve_relevant_docs(query, faiss_index, k=k2)

    # Merge preserving order, remove duplicates by chunk_id
    seen = set()
    merged = []
    for doc in bm25_docs + emb_docs:
        cid = doc.metadata["chunk_id"]
        if cid not in seen:
            seen.add(cid)
            merged.append(doc)

    # Cross‑encode & rerank the merged set
    reranked = rerank_documents(query, merged, cross_encoder)

    # Return top‑final_k
    return reranked[:final_k]

In [44]:
def create_audio_summary(audio_features):
    """
    Create a textual summary from the engineered audio features.
    """
    pitch_z = audio_features.get('pitch_z', 'N/A')
    rms_z = audio_features.get('rms_z', 'N/A')
    composite_intensity = audio_features.get('composite_intensity', 'N/A')

    summary = (f"\nAudio Analysis Summary: The normalized pitch is {pitch_z:.4f} and "
               f"the normalized RMS energy is {rms_z:.4f}, leading to a composite intensity of "
               f"{composite_intensity:.4f}.")
    return summary


def build_multimodal_prompt(retrieved_docs, query, prompt_engineering):
    context_chunks = []

    for i, doc in enumerate(retrieved_docs, start=1):
        # Use only speaker metadata now
        speaker = doc.metadata.get('speaker', 'Unknown')
        source_info = f"\n[Source: {speaker}]"

        # Use question if it exists
        question = doc.metadata.get("question", "")
        if question:
            chunk_text = f"Q: {question}\nA: {doc.page_content}"
        else:
            chunk_text = doc.page_content

        # Incorporate audio features (if available) from metadata
        audio_feat = doc.metadata.get("audio_features", None)
        if audio_feat:
            # Build a summary of the audio features. Adjust keys if necessary.
            # pitch_mean = audio_feat.get("pitch_mean", "N/A")
            # rms_mean = audio_feat.get("rms_mean", "N/A")
            # composite_intensity = audio_feat.get("composite_intensity", "N/A")
            # audio_summary = (f"\n[Audio Features: Pitch Mean: {pitch_mean}, "
            #                  f"RMS Mean: {rms_mean}, "
            #                  f"Composite Intensity: {composite_intensity}]")
            audio_summary = create_audio_summary(audio_feat)
        else:
            audio_summary = ""

        # Replace newlines with spaces
        chunk_text = chunk_text.replace("\n", " ")

        # Append the formatted chunk text with speaker and audio info
        context_chunks.append(f"{i}. {chunk_text}{source_info}{audio_summary}")

    # Join all context chunks
    context = "\n\n".join(context_chunks)

    # Build the final multimodal prompt (including instructions and query)
    prompt = (
        "You are a financial analyst reviewing an earnings call transcript and its associated audio features. "
        "Your task is to extract and summarize key factual claims and sentiment-related insights expressed by management "
        "that can later be verified against the company's 10-K filing. The 10-K contains detailed data in sections such as "
        "the Management Discussion and Analysis (MD&A), Financial Statements, Business Overview, and Risk Factors.\n\n"
        "IMPORTANT: Take into account the audio features (like pitch, RMS energy, "
        "and composite intensity) in your elaboration to provide a comprehensive analysis.\n\n"

        "Instructions:\n\n"
        "Based on the retrieved context from earnings call transcripts (with their corresponding audio features) below:\n\n"
        "#### Context/Retrieved Statements:\n{context}\n\n"
        "Identify and extract factual claims regarding the company’s performance, outlook, and any sentiment indications "
        "(for example, optimism about growth, caution regarding risks, or confidence in future performance).\n\n"
        "For each claim, provide the following details in a structured format:\n\n"
        "Claim Text: A concise statement of the fact or sentiment.\n\n"
        "[Source: speaker]\n\n"
        "Metric/Detail (if applicable): Any quantifiable data (e.g., 'revenue growth of 15%', 'EBITDA margin improvement').\n\n"
        "Relevant Reporting Period: Indicate the fiscal year or quarter mentioned (e.g., 'FY2023').\n\n"
        "Target 10-K Section: Suggest which section of the 10-K is most appropriate to verify this claim (for example, 'MD&A', 'Financial Statements', 'Risk Factors', 'Business Overview').\n\n"
        "#### Query: {query}\n\n"
    ).format(context=context, query=query, prompt_engineering=prompt_engineering)

    return prompt


prompt_engineering = ""

query = "Analyze Apples's sentiment level."
# Retrieve using our hybrid BM25 + FAISS and rerank documents using cross encoder
retrieved_docs = hybrid_retrieve(query, k1=5, k2=5, final_k=5, vector_store=faiss_index, cross_encoder=cross_encoder)


# Build the prompt using the retrieved documents
prompt = build_multimodal_prompt(retrieved_docs, query, prompt_engineering)
md("======================== Constructed Prompt: ========================")
md(prompt)

======================== Constructed Prompt: ========================

You are a financial analyst reviewing an earnings call transcript and its associated audio features. Your task is to extract and summarize key factual claims and sentiment-related insights expressed by management that can later be verified against the company's 10-K filing. The 10-K contains detailed data in sections such as the Management Discussion and Analysis (MD&A), Financial Statements, Business Overview, and Risk Factors.

IMPORTANT: Take into account the audio features (like pitch, RMS energy, and composite intensity) in your elaboration to provide a comprehensive analysis.

Instructions:

Based on the retrieved context from earnings call transcripts (with their corresponding audio features) below:

#### Context/Retrieved Statements:
1. Q: Great. Thanks so much for taking my questions. I have two as well. Tim, maybe if we start with you, I think each of the last four years, you've exited the December quarter with iPhone demand outpacing supply. As we look to this quarter in the iPhone 16 cycle, lead times are relatively short. There are no known supply shortages. And I'm just curious whether you've been able to maybe get a better read on early cycle iPhone demand this year relative to past years and if so, what you've learned about upgrade rates, switching rates, trade-ups versus trading down, and being more price sensitive. And overall, any impact that Apple Intelligence may have on iPhone 16 sales? And then I have a follow-up. Thank you. A: There's a lot there. On Apple Intelligence, we believe it's a compelling upgrade reason. And we'll -- but we just launched it three days ago, and so what we've got now from a data point of view is the number I just referenced that 18.1 has twice the adoption rate of 17.1. So, that clearly shows a level of interest out there. In terms of exiting the December quarter with demand greater than supply, that's not my recollection that that happened for all four of the years. We clearly had cases during COVID where there were disruptions, and that's -- some spilled over. But in a more regular environment where we're not having something, a 100-year flood kind of thing, we would -- our desire is to get into balance as quickly as possible. We don't want customers having to wait for products. And so, if you look at how we've done this year, we did that very quickly on the 16, on the 16 Pro family, the Pro and the Pro Max. We've been constrained in October, but we believe that soon we'll be out of constraint. And so, that's a good sign from our point of view. Keep in mind that that's a function of supply and demand, not one side or the other. And we've been preparing for the quarter for a while. So, that's what I would say there.
[Source: Timothy Donald Cook -- Chief Executive Officer]
Audio Analysis Summary: The normalized pitch is 0.0212 and the normalized RMS energy is 0.8078, leading to a composite intensity of 0.8290.

2. Q: Hey, thanks a lot. And I'll echo those comments about Luca. We'll miss you and good luck. And my question is with regard to iPhone again. And with regard to the fourth quarter is my first question -- or sorry, the fourth calendar quarter, your first quarter. When you look at mid- to low single-digit revenue growth, do you expect the iPhone to grow faster? And what are you thinking about in the answer to that question with regard to China, which keeps improving each quarter? And then I have just a follow-up. Thanks. A: Yeah. You know, Ben, we are not providing that level of color today. Yes, we've said that we expect total company revenue to grow low to mid-single digits. Keep in mind, Apple Intelligence, as Tim said, is rolling out over time, both features and languages. And we just had a number of exciting launches just this week from the Apple Intelligence feature to the new Mac. So, we'll leave it at that. We've given you the total for the company and some pretty good direction on services, which we expect to continue to grow at a similar rate than what we've seen in fiscal '24.
[Source: Luca Maestri -- Senior Vice President, Chief Financial Officer]
Audio Analysis Summary: The normalized pitch is -0.3393 and the normalized RMS energy is 1.3286, leading to a composite intensity of 1.6679.

3. Thank you, Suhasini. Good afternoon, everyone, and thanks for joining the call. Today, Apple is reporting revenue of $94.9 billion, a September quarter record and up 6% from a year ago. iPhone grew in every geographic segment, marking a new September quarter revenue record for the category, and services set an all-time revenue record, up 12% year over year. We also set September quarter segment revenue records in the Americas, Europe, and the rest of Asia Pacific as well as in a large number of countries, including the United States, Brazil, Mexico, France, the U.K., Korea, Malaysia, Thailand, Saudi Arabia, and the UAE. And we continue to be excited by the enthusiasm we're seeing in India, where we set an all-time revenue record. This has been an extraordinary year of innovation at Apple. We brought the revolutionary Apple Vision Pro to customers in February, which brings users tomorrow's technology today. And in June, we announced Apple Intelligence, a remarkable personal intelligent system that combines the power of generative models with personal context to deliver intelligence that is incredibly useful and relevant. Apple Intelligence marks the beginning of a new chapter for Apple Innovation and redefines privacy and AI by extending our groundbreaking approach to privacy into the cloud with private cloud compute. Earlier this week, we made the first set of Apple Intelligence features available in U.S. English for iPhone, iPad, and Mac users with systemwide writing tools that help you refine your writing, a more natural and conversational Siri, a more intelligent Photos app, including the ability to create movies simply by typing a description, and new ways to prioritize and stay in the moment with notification summaries and priority messages. And we look forward to additional intelligence features in December with even more powerful writing tools, a new visual intelligence experience that builds on Apple Intelligence and ChatGPT integration as well as localized English in several countries, including the U.K., Australia, and Canada. These features have already been provided to developers, and we're getting great feedback. More features will be rolling out in the coming months as well as support for more languages, and this is just the beginning. Now, I'll turn to our results for the quarter, beginning with iPhone. iPhone revenues set a September quarter record of $46.2 billion, up 6% from a year ago with growth in every geographic segment. With the introduction of Apple Intelligence, we're beginning a new era for iPhone. iPhone 16 powered by 18 is equipped with an incredible new 48-megapixel Fusion camera, fantastic photo experiences, and the addition of the action button and camera control. And iPhone 16 Pro is the most advanced iPhone we've ever made, powered by A18 Pro and featuring even larger displays, an industry-leading pro camera system with camera control, and studio-quality mics, all with a huge leap in battery life. Turning to Mac. Revenue was $7.7 billion, up 2% from a year ago. Just this week, we brought a new generation of Apple silicon to Mac, M4, M4 Pro, and M4 Max. From blazing-fast performance to Apple's most advanced neural engine yet, our latest chips can easily tackle incredibly complex workflows. And they ensure our newest Macs will be the best personal computers for AI the instant they hit stores. With the newest additions to our Mac lineup, customers can choose the Mac that's just right for them. Whether that's iMac, the world's best and most beautiful all-in-one; MacBook Air, the world's most popular laptop now with double the starting memory; MacBook Pro, the best pro notebook anywhere; or the incredible mighty new Mac mini, our first-ever carbon-neutral Mac. iPad revenue was $7 billion, 8% higher year over year. iPad is unlike any other product on the market today, and it's become an essential device in homes, schools, and businesses of all sizes. Recently, we were thrilled to introduce the newest iPad mini featuring an ultra-compact design built for Apple Intelligence with support for Apple Pencil Pro. It's been a big year for iPad. iPad Air was popular with students and teachers as they got back to school this year, while creators are pushing the boundaries of what's possible with the M4-powered iPad Pro. In Wearables, Home, and Accessories, revenue was $9 billion, down 3% from a year ago. During the quarter, we launched the all-new Apple Watch Series 10, bringing a beautiful new design and new capabilities to the world's most popular watch that make it even more powerful, intelligent, and sophisticated. It's the thinnest Apple Watch yet, making it more comfortable than ever while offering the biggest, most advanced display. watchOS 11 brings some huge new health and fitness insights to users including sleep apnea notifications, which help to alert people with a potentially serious but often undiagnosed condition. We're proud of the impact we make through our health innovations on Watch, and I'm grateful for every note I receive about the importance of watching people's lives. With AirPods 4, we've broken new ground in comfort and design with our best-ever open-ear headphones available for the first time with active noise cancellation. And we were especially pleased to unveil revolutionary end-to-end hearing health capabilities for AirPods Pro 2 with hearing protection, hearing test, and hearing aid features. These just became available in a software update this week, and we believe this will make a meaningful difference in our users' lives. I've already started getting notes from customers calling the experience life-changing. And Apple Vision Pro continues to deliver special experiences that weren't possible before, including immersive entertainment like the new short film, Submerged, which gives people a view into the unique storytelling power made possible by spatial computing. Vision Pro has more than 2,500 native spatial apps and 1.5 million compatible apps for visionOS 2 as well as applications companies are building to reimagine how they work. Vision Pro continues to inspire awe in its users, and we're just scratching the surface of what's possible. And just yesterday, we announced we're bringing Vision Pro to Korea and the UAE. As I mentioned earlier, Services achieved an all-time revenue record of $25 billion, up 12% from a year ago and with all-time revenue records across most of our categories. With Apple TV+, we love celebrating the craft of great storytellers who know how to put on a show. Audiences love to discover new movies like Wolfs, explore acclaimed new series like Disclaimer, and dive back into returning favorites like Slow Horses and Shrinking. Apple TV+ productions have become fixtures at award shows earning more than 2,300 nominations and more than 500 wins today. Apple also offers a live sports experience in a league of its own with MLS Season Pass, and subscribers have been cheering on their favorite teams in the MLS Cup Playoffs. This month, we also marked 10 years of Apple Pay. There's always something magical about being able to buy groceries or pay for movie tickets seamlessly with your Apple device. Today, users choose Apple Pay for purchases across tens of millions of retailers worldwide. And we're excited to make the Apple Pay experience even better with the option to redeem rewards and access loans from credit cards, debit cards, and other lenders right at checkout. Whenever we celebrate big moments, Apple Stores are the best places to share them with customers. I had an incredible time during launch day in September alongside our team at Apple Fifth Avenue where energy and enthusiasm filled the air. And in stores all over the world, customers are eager to get a closer look at our latest innovations. We also opened two new stores during the quarter, and we can't wait to bring four new stores to customers in India. We're passionate about education and believe technology has a vital role to play in both helping teachers to inspire their students and students to learn about the world around them. In honor of World Teachers' Day, Apple was proud to share new resources for teachers to engage their students in ways that aim to make learning easy and fun. Additionally, we've expanded our education grant program into 100 new schools and communities helping with everything from access to technology to educator resources to scholarships and financial support. As we near the end of the year, we're proud of the progress we've made in our efforts to be carbon-neutral across our entire footprint by the end of the decade. As I mentioned earlier, we were thrilled to introduce our first-ever carbon-neutral Mac with the latest Mac mini. And in another milestone, customers can choose a carbon-neutral option of any Apple Watch. These achievements are amazing for all of us at Apple, and we are determined to reach our 2030 goal. At Apple, across everything we do, we manage for the long term because we're always thinking about what comes next, the next great challenge, the next innovative idea, the next big breakthrough. As we close out the year, we have the best lineup we've ever had going into the holiday season, including Apple Intelligence, which marks the start of a new chapter for our products. This is just the beginning of what we believe generative AI can do, and I couldn't be more excited for what's to come. Before I hand it over to Luca, with Luca transitioning to a new role with Apple, this will be the final time he's joining our call. So, I just wanted to take a moment to recognize his extraordinary service as Apple's CFO and to thank him for his partnership. I am deeply grateful. In his 10 years in the role, Luca has done truly exceptional work in shaping Apple as we know it today. He has helped manage Apple for the long term thoughtfully and deliberately. He has helped us enrich the lives of so many around the world, and he has been a leader that people look up to and have learned so much from. I have incredible confidence in our incoming CFO, Kevan Parekh, and we look forward to more of you meeting and working with him going forward. With that, I'll turn it over to Luca.
[Source: Timothy Donald Cook -- Chief Executive Officer]
Audio Analysis Summary: The normalized pitch is 0.6142 and the normalized RMS energy is 0.7871, leading to a composite intensity of 1.4013.

4. Q: Great. Thanks, everyone for taking my questions. And congratulations, Luca. I know, Luca, and I know, Tim, you don't want to give a lot of granularity. But if I just try to pull together your comments about what the demand environment looks like, are we to assume, based on sort of the commentary, that there is a risk that maybe the product revenue portfolio could be down in the December quarter, if I take your numbers at face value? And if that's the risk, is that more iPhone-related, Mac-related given the strength that you've seen in iPad-related? Just trying to get a handle on kind of what potentially is giving you that degree of, I don't want to say caution but maybe balanced for you going into the December quarter. And then I have a follow-up. A: As I said, David, we're not providing that level of color. We've given you some data on services. I would repeat what I said earlier. We're very early in the cycle, very early in the cycle with a lot of new products and features that we are launching. And we're very excited about them, but it's early. And the Apple Intelligence rollout is going to happen over time, not across the world as normally we do with software releases.
[Source: Luca Maestri -- Senior Vice President, Chief Financial Officer]
Audio Analysis Summary: The normalized pitch is -0.1808 and the normalized RMS energy is 1.1141, leading to a composite intensity of 1.2949.

5. Q: Right. OK. So, maybe a follow-up for Tim. When you think about, to Luca's point, about the rollout being staged over the next several quarters across the world, do you think that has any impact on sort of the normal historical demand cadence across different regions? So, should we see something different, let's say, in the December quarter, the March quarter, or the June quarter, etc., relative to history given the timing of the rollout and where customers are probably waiting for the devices to be enabled to have the operating system? Would just love to kind of get your perspective on how we think about the demand cadence, how it might be different than maybe historically. Thank you. A: Yeah, David. It's clearly, as you point out, a different cadence, if you will, than we would normally do. And we -- as we talked about at WWDC, we wanted to give a comprehensive vision of Apple Intelligence, and we said then that it would roll out over time, and we're right on the -- what we said at WWDC. And so, we're executing well. In terms of the demand curve, I would just say that what we believe here is that it's a compelling reason for upgrading. And it's -- that's both my personal experience and feedback that I'm getting. And so, we'll see. We're not projecting beyond the current quarter obviously. We just don't do that.
[Source: Timothy Donald Cook -- Chief Executive Officer]
Audio Analysis Summary: The normalized pitch is 0.0066 and the normalized RMS energy is 0.0641, leading to a composite intensity of 0.0707.

Identify and extract factual claims regarding the company’s performance, outlook, and any sentiment indications (for example, optimism about growth, caution regarding risks, or confidence in future performance).

For each claim, provide the following details in a structured format:

Claim Text: A concise statement of the fact or sentiment.

[Source: speaker]

Metric/Detail (if applicable): Any quantifiable data (e.g., 'revenue growth of 15%', 'EBITDA margin improvement').

Relevant Reporting Period: Indicate the fiscal year or quarter mentioned (e.g., 'FY2023').

Target 10-K Section: Suggest which section of the 10-K is most appropriate to verify this claim (for example, 'MD&A', 'Financial Statements', 'Risk Factors', 'Business Overview').

#### Query: Analyze Apples's sentiment level.



In [ ]:
pip install openai==1.75.0

In [3]:
openai.__version__

'1.75.0'

In [45]:
def generate_insight(prompt, model="gpt-4-turbo", temperature=0.1, max_tokens=1080):
    """
    Generate analysis using the given prompt via the OpenAI ChatCompletion API.
    """
    response = openai.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are interested in analyzing a company's sentiment level."},
            {"role": "user", "content": prompt}
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content


# Generate an insight using GPT
insight = generate_insight(prompt)
md("======================== Generated Analysis: ========================")
md(insight)

======================== Generated Analysis: ========================

### Analysis of Apple's Sentiment Level

Based on the earnings call transcript and audio analysis, the sentiment expressed by Apple's management can be summarized as cautiously optimistic with a strong focus on innovation and market expansion. The sentiment indicators are derived from the factual claims made by the speakers, their tone, and the audio features analyzed.

#### Sentiment Indicators:

1. **Optimism about Product Innovation and Market Expansion:**
   - **Claim Text:** Apple has introduced revolutionary products like Apple Vision Pro and Apple Intelligence, which are expected to redefine user experiences and privacy in AI.
   - **Source:** Timothy Donald Cook
   - **Metric/Detail:** Introduction of new products and features like Apple Vision Pro and Apple Intelligence.
   - **Relevant Reporting Period:** FY2023
   - **Target 10-K Section:** Business Overview

2. **Confidence in Revenue Growth and Geographic Expansion:**
   - **Claim Text:** Apple reported a revenue of $94.9 billion, a record for the September quarter, with growth in every geographic segment and services achieving an all-time revenue record.
   - **Source:** Timothy Donald Cook
   - **Metric/Detail:** Revenue of $94.9 billion, growth in all geographic segments.
   - **Relevant Reporting Period:** Q3 FY2023
   - **Target 10-K Section:** Financial Statements

3. **Positive Outlook on Future Product Rollouts:**
   - **Claim Text:** Apple Intelligence features are rolling out over time, with more powerful tools and additional language support planned, indicating a staged but continuous innovation strategy.
   - **Source:** Timothy Donald Cook
   - **Metric/Detail:** Staged rollout of Apple Intelligence features.
   - **Relevant Reporting Period:** FY2023 and beyond
   - **Target 10-K Section:** MD&A

4. **Cautious Optimism Regarding Supply and Demand Balance:**
   - **Claim Text:** Apple aims to quickly balance supply and demand, especially highlighted by the quick resolution of constraints for the iPhone 16 Pro family.
   - **Source:** Timothy Donald Cook
   - **Metric/Detail:** Quick resolution of supply constraints for iPhone 16 Pro.
   - **Relevant Reporting Period:** Q4 FY2023
   - **Target 10-K Section:** MD&A

5. **Concerns Over Demand and Revenue in the Short Term:**
   - **Claim Text:** There is a cautious sentiment about the potential for product revenue to be down in the December quarter, reflecting uncertainty in the demand environment.
   - **Source:** Luca Maestri
   - **Metric/Detail:** Potential decrease in product revenue for the December quarter.
   - **Relevant Reporting Period:** Q4 FY2023
   - **Target 10-K Section:** Risk Factors

#### Audio Features Analysis:
- **Timothy Donald Cook's segments** showed varying levels of composite intensity, with the highest being 1.4013 during a detailed discussion of Apple's performance and innovations. This higher intensity could indicate strong engagement and confidence in the company's direction.
- **Luca Maestri's responses** had lower pitch but higher RMS energy, particularly when discussing financial expectations without providing detailed forecasts, reflecting a cautious but firm stance on financial matters.

### Conclusion:
The overall sentiment from Apple's management during the earnings call is cautiously optimistic, with a strong emphasis on continuous innovation and strategic market expansion. The management shows confidence in their new products and features while maintaining a cautious approach to immediate financial forecasts and supply-demand dynamics. This sentiment is crucial for stakeholders and can be further verified in the detailed sections of the upcoming 10-K filing, particularly in the MD&A and Financial Statements sections.

## Retrieval Eval

In [22]:
from fuzzywuzzy import fuzz

test_queries = {
    "Q1": "When did Apple announce its fourth-quarter 2024 financial results?",
    "Q2": "What was Apple's total revenue for Q4 2024, and how did it compare year-over-year?",
    "Q3": "Which product categories set revenue records in Q4 2024?",
    "Q4": "What was the performance of Apple's Services segment in Q4 2024?",
    "Q5": "What is Apple Intelligence, and how is it being rolled out?",
    "Q6": "How did iPhone revenue perform in Q4 2024?",
    "Q7": "What were the key highlights of Apple's Wearables, Home, and Accessories segment?",
    "Q8": "What dividend did Apple declare for Q4 2024?",
    "Q9": "What was Apple's outlook for the December quarter (Q1 2025)?",
    "Q10": "What were the key health features announced for Apple Watch and AirPods?"
}

ground_truths = {
    "Q1": "Apple announced its Q4 2024 financial results on October 31, 2024.",
    "Q2": "Apple reported Q4 2024 revenue of $94.9 billion, up 6% year-over-year.",
    "Q3": "iPhone and Services set all-time revenue records for the September quarter, with segment records in the Americas, Europe, and rest of Asia Pacific.",
    "Q4": "Services revenue reached an all-time record of $25 billion, up 12% year-over-year, with growth across most categories.",
    "Q5": "Apple Intelligence is a personal AI system combining generative models with personal context. It is being rolled out in phases, starting with U.S. English in "
    "October 2024, expanding to more languages and features in December 2024 and beyond.",
    "Q6": "iPhone revenue was $46.2 billion, up 6% year-over-year, setting a September quarter record.",
    "Q7": "Apple Watch Series 10 introduced sleep apnea notifications, and AirPods Pro 2 added hearing health features like hearing tests and hearing aid capabilities.",
    "Q8": "Apple declared a cash dividend of $0.25 per share, payable on November 14, 2024.",
    "Q9": "Apple expects Q1 2025 revenue to grow low to mid-single digits year-over-year, with Services growing at a similar rate to fiscal 2024.",
    "Q10": "Apple Watch added sleep apnea notifications, while AirPods Pro 2 introduced hearing test and hearing aid features, described as 'life-changing' by users."
}

# Number of top results to retrieve
TOP_K = 5


# --- Evaluation Metrics ---

def fuzzy_match(text1, text2, threshold=50):
    """Returns True if two texts have high similarity (fuzzy match)."""
    return fuzz.partial_ratio(text1.lower(), text2.lower()) > threshold

def recall_at_k(retrieved_docs, relevant_doc, k):
    """Checks if relevant_doc is in top-K retrieved docs using fuzzy matching."""
    return int(any(fuzzy_match(relevant_doc, doc) for doc in retrieved_docs[:k]))

def precision_at_k(retrieved_docs, relevant_doc, k):
    """Fraction of retrieved docs that contain the relevant passage (fuzzy match)."""
    relevant_count = sum(1 for doc in retrieved_docs[:k] if fuzzy_match(relevant_doc, doc))
    return relevant_count / k if k > 0 else 0

def reciprocal_rank(retrieved_docs, relevant_doc):
    """Finds the first occurrence of relevant_doc using fuzzy matching."""
    for rank, doc in enumerate(retrieved_docs, start=1):
        if fuzzy_match(relevant_doc, doc):
            return 1 / rank
    return 0

c:\Users\Goh Ming Wee\AppData\Local\Programs\Python\Python39\lib\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [23]:
import numpy as np

# Store retrieved results
retrieved_results = {}

# Query the retriever
for q_id, query in test_queries.items():
    retrieved_docs = hybrid_retrieve(query, k1=5, k2=5, final_k=5, vector_store=faiss_index, cross_encoder=cross_encoder)
    retrieved_texts = [doc.page_content for doc in retrieved_docs]
    retrieved_results[q_id] = retrieved_texts

# --- Compute Metrics ---
recall_scores = []
precision_scores = []
mrr_scores = []

for q_id in test_queries.keys():
    retrieved_docs = retrieved_results[q_id]
    relevant_doc = ground_truths[q_id]

    recall_scores.append(recall_at_k(retrieved_docs, relevant_doc, TOP_K))
    precision_scores.append(precision_at_k(retrieved_docs, relevant_doc, TOP_K))
    mrr_scores.append(reciprocal_rank(retrieved_docs, relevant_doc))

# Compute mean scores
mean_recall_at_k = np.mean(recall_scores)
mean_precision_at_k = np.mean(precision_scores)
mrr = np.mean(mrr_scores)

# Print Results
print(f"Evaluation Metrics for RAG Retriever:")
print(f"----------------------------------")
print(f"Recall@{TOP_K}: {mean_recall_at_k:.2f}")
print(f"Precision@{TOP_K}: {mean_precision_at_k:.2f}")
print(f"MRR (Mean Reciprocal Rank): {mrr:.2f}")

Evaluation Metrics for RAG Retriever:
----------------------------------
Recall@5: 0.50
Precision@5: 0.16
MRR (Mean Reciprocal Rank): 0.20


In [154]:
openai.__version__

'1.17.0'

## Generation Eval

In [ ]:
# def evaluate_citation(generated_text, retrieved_docs):

#     # pattern = r"- \*\*Source:\*\*\s*(.+)"
#     # pattern = r"\[Source:\s*<([^>]+)>\]"
#     # pattern = re.compile(r'\[Source:\s*([^\-]+?)\s*--\s*([^\]]+?)\]')
#     pattern = re.compile(
#     r'Source:\s*(?P<source>.+)$',
#     re.MULTILINE)
#     citations = re.findall(pattern, generated_text)

#     if not citations:
#         print("No citations found in the generated text.")
#         return 0.0

#     matched = 0
#     for cited in citations:
#         # print(cited[0][2:])
#         # cited_speaker = cited.strip().split("--")[0].strip().lower()
#         cited_speaker = cited[0][2:].strip().lower()
#         # print(cited_speaker)

#         for doc in retrieved_docs:
#             doc_speaker = doc.metadata.get("speaker", "").split("--")[0].strip().lower()
#             # print(doc_speaker)

#             if cited_speaker == doc_speaker:
#                 matched += 1
#                 break

#     citation_score = matched / len(citations)
#     return citation_score

import re

def evaluate_citation(generated_text, retrieved_docs):
    """
    Compute the fraction of correctly cited speakers in generated_text
    against the speakers in retrieved_docs.
    """
    # Regex to capture "Source: Speaker Name[, Title]" at end of line
    pattern = re.compile(r'Source:\s*(?P<source>[^,\n]+)', re.MULTILINE)
    citations = pattern.findall(generated_text)  # list of speaker names

    if not citations:
        print("No citations found in the generated text.")
        return 0.0

    matched = 0
    # Precompute retrieved speaker names (normalized)
    retrieved_speakers = {
        doc.metadata.get("speaker", "").split("--")[0].strip().lower()
        for doc in retrieved_docs
    }

    for cited in citations:
        # Normalize cited speaker (strip whitespace, lowercase)
        cited_name = cited[3:].strip().lower()
        # print(cited_name)

        if cited_name in retrieved_speakers:
            matched += 1

    citation_score = matched / len(citations)
    return citation_score

# Generation evaluation using Groundedness Check
citation_score = evaluate_citation(insight, retrieved_docs)
md("### Generation Citation score:")
print(citation_score)

### Generation Citation score:

1.0


## Cosine similarity test

Computes the semantic similarity between a generated claim and a retrieved evidence passage using a pre-trained sentence transformer model. This verifies whether the generated content is truly supported by the retrieved documents, helping detect hallucinations and ensuring factual accuracy by grounding model outputs in actual source evidence.


In [71]:
# # Markdown output
# def md(text):
#     display(Markdown(text))

# # Load embedding model
# model = SentenceTransformer('all-MiniLM-L6-v2')

# def compute_cosine_similarity(claim, evidence):
#     embeddings = model.encode([claim, evidence], convert_to_tensor=True)
#     cosine_sim = util.cos_sim(embeddings[0], embeddings[1])
#     return cosine_sim.item()

# def evaluate_groundedness_cosine(generated_text, retrieved_docs, threshold=0.5):
#     """
#     Evaluates groundedness of each claim in generated output using cosine similarity.
#     Matches claims to retrieved docs via speaker metadata and compares to (question + answer) content.
#     """
#     # Extract all Claim Texts and Sources from the markdown format
# #     claim_pattern = r"- \*\*Claim Text:\*\*\s*(.+)"
# #     # source_pattern = r"- \*\*Source:\*\*\s*(.+)"
# #     # source_pattern = re.compile(r'\[Source:\s*([^\-]+?)\s*--\s*([^\]]+?)\]')
# #     claim_pattern = re.compile(
# #     r'Claim Text:\s*'        # literal “Claim Text:” plus any spaces
# #     r'(.+?)'                 # Group 1: minimally match one or more chars
# #     r'(?=\s*(?:\[Source:|\n|$))'  # lookahead for “[Source:”, newline, or end‐of‐string
# # )

#     claims = re.findall(claim_pattern, generated_text)
#     sources = re.findall(source_pattern, generated_text)

#     total_claims = len(claims)
#     supported_claims = 0
#     sim_scores = []

#     md("### Generation Groundedness Check:")

#     for claim_text, cited_speaker in zip(claims, sources):
#         # claim_text = claim_text.strip()
#         # cited_speaker = cited_speaker.split("--")[0].strip().lower()
#         claim_text = claim_text[2:].strip().lower()
#         cited_speaker = cited_speaker[0][2:].strip().lower()

#         # Match with the most relevant doc based on speaker
#         matched_evidence = None
#         for doc in retrieved_docs:
#             doc_speaker = doc.metadata.get("speaker", "").split("--")[0].strip().lower()
#             if cited_speaker == doc_speaker:
#                 question = doc.metadata.get("question", "").strip()
#                 answer = doc.page_content.strip()
#                 matched_evidence = f"Q: {question} A: {answer}" if question else answer
#                 break

#         if matched_evidence:
#             similarity = compute_cosine_similarity(claim_text, matched_evidence)
#             print(f"\nClaim: {claim_text}\nEvidence: {matched_evidence}\nSimilarity: {similarity:.4f}")
#             sim_scores.append(similarity)
#             if similarity >= threshold:
#                 supported_claims += 1
#         else:
#             print(f"\nClaim: {claim_text}\nNo matching evidence found for: {cited_speaker}")

#     groundedness_score = supported_claims / total_claims if total_claims > 0 else 0.0

#     md("#### Average Similarity Score:")
#     print(statistics.mean(sim_scores) if sim_scores else "N/A")

#     md("#### Generation Groundedness Score:")
#     print(f"{groundedness_score:.2f}")

#     return

# evaluate_groundedness_cosine(insight, retrieved_docs)




import re
import statistics
from sentence_transformers import SentenceTransformer, util
from IPython.display import display, Markdown

# Helper to render Markdown
def md(text):
    display(Markdown(text))

# 1) Regex patterns to extract claims and sources
#    - Claim Text: capture everything after 'Claim Text:' up to the next 'Source:' or end of string
claim_pattern = re.compile(
    r'Claim Text:\s*(?P<claim>.+?)(?=\s*Source:|$)',
    flags=re.DOTALL
)
#    - Source: capture the speaker name before any comma or end of line
source_pattern = re.compile(
    r'Source:\s*(?P<speaker>[^,\n]+)',
    flags=re.MULTILINE
)

# 2) Load SBERT model for embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')

def compute_cosine_similarity(claim: str, evidence: str) -> float:
    """
    Compute cosine similarity between two texts using SBERT.
    """
    emb = model.encode([claim, evidence], convert_to_tensor=True)
    sim = util.cos_sim(emb[0], emb[1])
    return sim.item()

def evaluate_groundedness_cosine(generated_text: str, retrieved_docs: list, threshold: float = 0.4):
    """
    For each 'Claim Text' and its corresponding 'Source', find the matching doc by speaker,
    compute cosine similarity against the doc's Q/A text, and report groundedness metrics.
    """
    # Extract all claims and sources
    claims = claim_pattern.findall(generated_text)
    speakers = source_pattern.findall(generated_text)
    total_claims = len(claims)
    supported_claims = 0
    sim_scores = []

    md("### Generation Groundedness Check")

    for claim, cited in zip(claims, speakers):
        # Normalize inputs
        claim_text = claim[3:-4].strip()
        cited_name = cited[3:].strip().lower()
        # print("--------------------------------")
        # print(claim_text)
        # print("--------------------------------")

        # Find matching evidence by speaker
        evidence = None
        for doc in retrieved_docs:
            doc_speaker = doc.metadata.get("speaker", "").split("--")[0].strip().lower()
            if doc_speaker == cited_name:
                question = doc.metadata.get("question", "").strip()
                answer = doc.page_content.strip()
                evidence = f"Q: {question} A: {answer}" if question else answer
                break

        if evidence:
            score = compute_cosine_similarity(claim_text, evidence)
            sim_scores.append(score)
            status = "✔" if score >= threshold else "✘"
            print(f"\nClaim: {claim_text}\nEvidence: {evidence}\nSimilarity: {score:.4f} {status}")
            if score >= threshold:
                supported_claims += 1
        else:
            print(f"\nClaim: {claim_text}\nNo evidence found for speaker: {cited_name}")

    # Aggregate and display results
    avg_sim = statistics.mean(sim_scores) if sim_scores else 0.0
    grounding_score = supported_claims / total_claims if total_claims > 0 else 0.0

    md("#### Average Similarity Score:")
    print(f"{avg_sim:.4f}")

    md("#### Groundedness Score:")
    print(f"{grounding_score:.2%}")



evaluate_groundedness_cosine(insight, retrieved_docs)

### Generation Groundedness Check


Claim: Apple has introduced revolutionary products like Apple Vision Pro and Apple Intelligence, which are expected to redefine user experiences and privacy in AI.
Evidence: Q: Great. Thanks so much for taking my questions. I have two as well. Tim, maybe if we start with you, I think each of the last four years, you've exited the December quarter with iPhone demand outpacing supply. As we look to this quarter in the iPhone 16 cycle, lead times are relatively short. There are no known supply shortages. And I'm just curious whether you've been able to maybe get a better read on early cycle iPhone demand this year relative to past years and if so, what you've learned about upgrade rates, switching rates, trade-ups versus trading down, and being more price sensitive. And overall, any impact that Apple Intelligence may have on iPhone 16 sales? And then I have a follow-up. Thank you. A: There's a lot there. On Apple Intelligence, we believe it's a compelling upgrade reason. And we'll -- but

#### Average Similarity Score:

0.4430


#### Groundedness Score:

60.00%
